# 04 — Evaluation, Threshold Analysis and Error Review

Evaluate the final Random Forest on validation/test data. The 0.20 threshold is selected using validation results and then frozen for the held-out test set.

In [ ]:
import pandas as pd, joblib, numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,roc_curve
FEATURE_BASE = ['radius','texture','perimeter','area','smoothness','compactness','concavity','concave_points','symmetry','fractal_dimension']
FEATURES = [f'{stat}_{feat}' for stat in ['mean','se','worst'] for feat in FEATURE_BASE]
RANDOM_STATE=42
THRESHOLD=.20
df=pd.read_csv('data/wdbc.data',header=None,names=['id','diagnosis']+FEATURES)
X=df[FEATURES]; y=(df.diagnosis=='M').astype(int)
X_train,X_temp,y_train,y_temp=train_test_split(X,y,test_size=.30,stratify=y,random_state=RANDOM_STATE)
X_val,X_test,y_val,y_test=train_test_split(X_temp,y_temp,test_size=.50,stratify=y_temp,random_state=RANDOM_STATE)
model=joblib.load('models/rf_model.joblib')

In [ ]:
val_prob=model.predict_proba(X_val)[:,1]
val_pred=(val_prob>=THRESHOLD).astype(int)
print('Validation metrics at threshold 0.20')
print('Precision:',precision_score(y_val,val_pred),'Recall:',recall_score(y_val,val_pred),'F1:',f1_score(y_val,val_pred),'AUC:',roc_auc_score(y_val,val_prob))

In [ ]:
test_prob=model.predict_proba(X_test)[:,1]
test_pred=(test_prob>=THRESHOLD).astype(int)
print('Test metrics')
print('Accuracy:',accuracy_score(y_test,test_pred))
print('Precision:',precision_score(y_test,test_pred))
print('Recall(M):',recall_score(y_test,test_pred))
print('F1:',f1_score(y_test,test_pred))
print('ROC-AUC:',roc_auc_score(y_test,test_prob))
print('Confusion matrix:\n',confusion_matrix(y_test,test_pred))

In [ ]:
fpr,tpr,_=roc_curve(y_test,test_prob)
plt.plot(fpr,tpr,label=f'AUC={roc_auc_score(y_test,test_prob):.3f}'); plt.plot([0,1],[0,1],'--'); plt.xlabel('False Positive Rate'); plt.ylabel('True Positive Rate'); plt.title('ROC Curve'); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
cm=confusion_matrix(y_test,test_pred)
print('TN, FP, FN, TP:',cm.ravel())
errors=df.loc[X_test.index].copy()
errors['prob_M']=test_prob
errors['pred']=['M' if p else 'B' for p in test_pred]
print('False positives:')
print(errors[(errors.diagnosis=='B')&(errors.pred=='M')][['id','prob_M']])
print('False negatives:')
print(errors[(errors.diagnosis=='M')&(errors.pred=='B')][['id','prob_M']])